# TARDIS
This notebook loads the dataset, cleans the data, handles missing values, and creates new features for further analysis.

In [ ]:
import pandas as pd
import numpy as np

## Load and Inspect the Dataset

In [ ]:
df = pd.read_csv("dataset.csv", sep=";")
print("(Nb rows, Nb columns): ", df.shape)
df.head()
df.info()
print(df.columns)

In [ ]:
nb_duplicates = df.duplicated().sum()

if nb_duplicates > 0:
    df = df.drop_duplicates()

print("Nb duplicates: ", nb_duplicates)
print("(Nb rows, Nb columns): ", df.shape)

The dataset is first checked for duplicate rows to avoid counting the same observation more than once in later analysis.

## Data Cleaning

In [ ]:
df["Date"] = df["Date"].str.replace("/", "-")
df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m", errors="coerce")

In [ ]:
numeric_cols = [
    "Average journey time",
    "Number of scheduled trains",
    "Number of cancelled trains",
    "Number of trains delayed at departure",
    "Average delay of late trains at departure",
    "Average delay of all trains at departure",
    "Number of trains delayed at arrival",
    "Average delay of late trains at arrival",
    "Average delay of all trains at arrival",
    "Number of trains delayed > 15min",
    "Average delay of trains > 15min (if competing with flights)",
    "Number of trains delayed > 30min",
    "Number of trains delayed > 60min",
    "Pct delay due to external causes",
    "Pct delay due to infrastructure",
    "Pct delay due to traffic management",
    "Pct delay due to rolling stock",
    "Pct delay due to station management and equipment reuse",
    "Pct delay due to passenger handling (crowding, disabled persons, connections)",
]

for col in numeric_cols:
    df[col] = df[col].astype(str).str.replace(",", ".")
    df[col] = pd.to_numeric(df[col], errors="coerce")

df[numeric_cols] = df[numeric_cols].fillna(0)

df.info()

Several columns that contain numeric values are converted into numeric format so they can be used for calculations, comparisons, and visual analysis.

In [ ]:
print(df.isnull().sum())

The dataset is checked for missing values so that incomplete rows or invalid values can be handled before feature creation.

In [ ]:
df = df.replace(["nan", "NaN", "None", "NULL"], np.nan)

Convert missing strings intro real NaN values.

In [ ]:
df["Service"] = df["Service"].fillna("Unknown").str.strip()
df["Departure station"] = df["Departure station"].fillna("Unknown").str.strip()
df["Arrival station"] = df["Arrival station"].fillna("Unknown").str.strip()

df = df.dropna(subset=["Date", "Average delay of all trains at arrival"])
df = df.drop_duplicates(
    subset=["Date", "Departure station", "Arrival station"], keep="first"
)
print("(Nb rows, Nb columns): ", df.shape)

Rows with missing values in the key columns are removed because these fields are necessary for reliable analysis and later feature engineering.

After handling missing values, the dataset is checked again for duplicates to avoid counting the same observation more than once.

## Feature Engineering

In this section, temporal features such as year, month, quarter, and season are added, together with route and delay-related rate features.

In [ ]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month

In [ ]:
df["Month name"] = df["Date"].dt.month_name()
df["Quarter"] = df["Date"].dt.quarter

In [ ]:
df["Season"] = df["Month"].map(
    {
        12: "Winter",
        1: "Winter",
        2: "Winter",
        3: "Spring",
        4: "Spring",
        5: "Spring",
        6: "Summer",
        7: "Summer",
        8: "Summer",
        9: "Autumn",
        10: "Autumn",
        11: "Autumn",
    }
)

In [ ]:
df["Route"] = df["Departure station"] + " -> " + df["Arrival station"]

In [ ]:
df["Total trains"] = df["Number of scheduled trains"]

df["Cancellation rate"] = df["Number of cancelled trains"] / df["Total trains"].replace(
    0, np.nan
)

df["Departure delay rate"] = df["Number of trains delayed at departure"] / df[
    "Total trains"
].replace(0, np.nan)

df["Arrival delay rate"] = df["Number of trains delayed at arrival"] / df[
    "Total trains"
].replace(0, np.nan)

In [ ]:
rate_cols = ["Cancellation rate", "Departure delay rate", "Arrival delay rate"]
df[rate_cols] = df[rate_cols].fillna(0)

df["Delay severity"] = pd.cut(
    df["Arrival delay rate"],
    bins=[-0.1, 0.1, 0.2, 1.0],
    labels=["Low", "Medium", "High"],
)

df[
    [
        "Year",
        "Month",
        "Month name",
        "Quarter",
        "Season",
        "Route",
        "Cancellation rate",
        "Departure delay rate",
        "Arrival delay rate",
        "Delay severity",
    ]
].head()

In [ ]:
df.to_csv("cleaned_dataset.csv", index=False)

## Arrival Delay Analysis

In [ ]:
df["Average delay of all trains at arrival"].describe()

In [ ]:
print("Mean arrival delay:", df["Average delay of all trains at arrival"].mean())
print("Median arrival delay:", df["Average delay of all trains at arrival"].median())
print("Minimum arrival delay:", df["Average delay of all trains at arrival"].min())
print("Maximum arrival delay:", df["Average delay of all trains at arrival"].max())

In [ ]:
df["Average delay of all trains at arrival"].plot(
    kind="hist", bins=10, title="Distribution of Average Arrival Delay"
)

This section analyzes the distribution of Average delay of all trains at arrival

In [ ]:
df["Average delay of all trains at arrival"].plot(
    kind="box", title="Boxplot of Average Arrival Delay"
)

In [ ]:
df.sort_values("Average delay of all trains at arrival", ascending=False).head()

## Average Arrival Delay by Departure Station

In [ ]:
departure_arrival_delay = (
    df.groupby("Departure station")["Average delay of all trains at arrival"]
    .mean()
    .sort_values(ascending=False)
)
departure_arrival_delay

In [ ]:
departure_arrival_delay.head(10).plot(
    kind="barh",
    figsize=(10, 6),
    title="Top 10 Departure Stations by Average Arrival Delay",
)

This analysis compares the top 10 departure stations with the highest average arrival delay.